# ASK, FSK, PSK, and QAM

This notebook introduces common digital modulation families by mapping a short bit pattern into amplitude, frequency, phase, or combined amplitude/phase changes.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from rf_utils import *
from IPython.display import Audio, Markdown, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

%matplotlib widget

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


## One Bitstream, Many Encodings

Digital modulation is just a decision about what physical parameter will represent symbols.

In [ ]:
bits = np.array([1, 0, 1, 1, 0, 0, 1, 0])
symbol_rate = 200
fs = 20_000
samples_per_symbol = fs // symbol_rate
t_symbol = np.arange(samples_per_symbol) / fs

def build_ask():
    return np.concatenate([(0.3 + 0.7 * bit) * np.cos(2 * np.pi * 1200 * t_symbol) for bit in bits])

def build_fsk():
    return np.concatenate([np.cos(2 * np.pi * (900 if bit == 0 else 1500) * t_symbol) for bit in bits])

def build_bpsk():
    return np.concatenate([np.cos(2 * np.pi * 1200 * t_symbol + (0 if bit == 1 else np.pi)) for bit in bits])

def build_qpsk():
    pairs = bits.reshape(-1, 2)
    phases = {(0, 0): 5 * np.pi / 4, (0, 1): 3 * np.pi / 4, (1, 1): np.pi / 4, (1, 0): 7 * np.pi / 4}
    return np.concatenate([np.cos(2 * np.pi * 1200 * t_symbol + phases[tuple(pair)]) for pair in pairs])

signals = {
    "ASK": build_ask(),
    "FSK": build_fsk(),
    "BPSK": build_bpsk(),
    "QPSK": build_qpsk(),
}


In [ ]:
audio_out = audio_output_widget()
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))

def update_scheme(scheme="ASK"):
    sig = signals[scheme]
    axes[0].clear()
    axes[1].clear()
    plot_waveform(sig[:2000], fs=fs, ax=axes[0], title=f"{scheme} waveform")
    plot_spectrum(sig, fs=fs, ax=axes[1], title=f"{scheme} spectrum")
    axes[1].set_xlim(0, 4000)
    axes[1].set_ylim(-100, 5)
    fig.canvas.draw_idle()
    refresh_audio_widget(audio_out, normalize(sig), rate=fs)

controls = widgets.interactive(
    update_scheme,
    scheme=dropdown(options=list(signals.keys()), value="ASK", description="Scheme"),
)
display(controls, audio_out)


## Key Takeaway

Digital modulation is still modulation. The difference is that the transmitter chooses from a finite symbol alphabet instead of letting the parameter vary continuously.